<!-- eds-seminar-variant -->
<div class="cd-kicker">EDS SEMINAR • PARTICIPANT NOTEBOOK</div>

[Download this participant notebook](participant.ipynb?download=1) · [Open the complete worked version](answers.ipynb)
<!-- /eds-seminar-variant -->

<div class="cd-kicker">CUBEDYNAMICS • RC3 BETA TESTER SUPERSHOWCASE</div>

<div class="cd-hero">
<h1>Four environmental stories, one analytical grammar</h1>
<p>This notebook is designed for a live room. It starts with recognizable observations, moves into scientific states and relationships, and repeatedly asks whether the visible code still reads like the scientific question.</p>
</div>

<div class="cd-grammar">noun → source → pipe → verb → state → trace → evidence</div>

### The four stories

**Story 1 · Working Lands**  
When and where did unusual warmth and little or no rain coincide?

**Story 2 · Boulder Cold Snap**  
Can a condition become an event, and can events reveal spatial synchrony and lag?

**Story 3 · Multivariate Weather**  
Can one grammar move across temperature, VPD, wind, humidity, precipitation, and radiation without erasing their differences?

**Story 4 · Remote Sensing**  
Can the same vocabulary reach from climate cubes into Sentinel-2 vegetation observations?

<div class="cd-card gold"><div class="cd-title">Visual rule for the talk</div>
Every story should contain at least one ordinary scientific figure and at least one <b>CubeDynamics-rendered interactive object</b>. Static figures explain the science. HTML cubes make the dimensional structure tangible.
</div>


## 0 · Install the exact release candidate

This install is pinned to the public commit tagged `v0.1.0rc3` and requires an
internet connection. If this kernel previously imported another CubeDynamics
version, restart it after installation and resume at section 1.


In [ ]:
%pip install -q --upgrade "cubedynamics @ git+https://github.com/CU-ESIIL/cubedynamics.git@f777eb07d3ada8bd407b027f560727c90f6d3731"


<div class="cd-card gold"><div class="cd-title">Before continuing</div>
If the package was updated in an already-running kernel, restart once and then run from setup downward.
</div>

In [ ]:
import sys, traceback, warnings, html
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
from IPython.display import display, HTML

import cubedynamics as cd
from cubedynamics import data, pipe, verbs as v

warnings.filterwarnings("ignore", category=FutureWarning)

display(HTML("""
<style>
:root{
  --cd-ink:#203039; --cd-muted:#5c6d75; --cd-blue:#2f7f9b; --cd-blue-bg:#eef6f8;
  --cd-green:#4f8b63; --cd-green-bg:#f5faf6; --cd-gold:#c8942f; --cd-gold-bg:#fffaf0;
  --cd-warm:#c65a37; --cd-warm-bg:#fff7f2; --cd-red:#b94a48; --cd-red-bg:#fff6f6;
}
.jp-Notebook,.notebook-container{max-width:1180px}
.cd-kicker{text-transform:uppercase;letter-spacing:.12em;font-weight:800;color:#55717d;font-size:12px;margin:4px 0 8px}
.cd-hero{border:1px solid #d6e2e7;border-radius:16px;padding:24px 26px;margin:12px 0 22px;background:linear-gradient(135deg,#f7fbfc,#fff)}
.cd-hero h1{margin:0 0 8px;font-size:34px;line-height:1.08;color:var(--cd-ink)}
.cd-hero p{font-size:17px;line-height:1.5;color:#41545e;margin:8px 0}
.cd-card{border:1px solid #d9e3e8;border-left:6px solid var(--cd-blue);border-radius:12px;padding:14px 18px;margin:12px 0;background:#f8fbfc}
.cd-card.warm{border-left-color:var(--cd-warm);background:var(--cd-warm-bg)}
.cd-card.green{border-left-color:var(--cd-green);background:var(--cd-green-bg)}
.cd-card.gold{border-left-color:var(--cd-gold);background:var(--cd-gold-bg)}
.cd-card.red{border-left-color:var(--cd-red);background:var(--cd-red-bg)}
.cd-title{font-weight:800;font-size:17px;margin-bottom:4px;color:var(--cd-ink)}
.cd-small{font-size:13px;color:var(--cd-muted)}
.cd-question{padding:16px 18px;border-radius:12px;margin:14px 0;background:var(--cd-blue-bg);border:1px solid #cde1e7;font-size:16px}
.cd-scroll{max-height:280px;overflow:auto;padding:12px 14px;border:1px solid #dbe4e8;border-radius:10px;background:#fbfcfd;font-family:ui-monospace,SFMono-Regular,Menlo,monospace;font-size:12px;white-space:pre-wrap}
.cd-grammar{text-align:center;font-size:20px;font-weight:800;padding:16px;border-radius:12px;background:#f7f9fa;border:1px solid #dde5e8;margin:16px 0}
.dataframe{font-size:13px}
</style>
"""))

RESULTS, FAILURES = {}, {}

def panel(title, body="", tone="green"):
    display(HTML(f'<div class="cd-card {tone}"><div class="cd-title">{html.escape(str(title))}</div><div>{body}</div></div>'))

def text_panel(title, text, tone=""):
    display(HTML(f'<div class="cd-card {tone}"><div class="cd-title">{html.escape(str(title))}</div><div class="cd-scroll">{html.escape(str(text))}</div></div>'))

def semantic_report(p):
    text_panel("What CubeDynamics says this analysis means", p.explain())
    try:
        text_panel("Semantic validation", p.validate(), "gold")
    except Exception:
        pass

def safe(name, fn):
    try:
        out = fn()
        panel(name, "Completed successfully.", "green")
        RESULTS[name] = out
        return out
    except Exception as exc:
        FAILURES[name] = f"{type(exc).__name__}: {exc}"
        panel(name, f"<b>{type(exc).__name__}</b>: {html.escape(str(exc))}", "red")
        return None

def state_field(obj):
    if isinstance(obj, xr.Dataset):
        if "state" in obj: return obj["state"]
        if len(obj.data_vars)==1: return obj[next(iter(obj.data_vars))]
    return obj

def compact_attrs(obj):
    keys=["scientific_noun","source","source_product","source_variable","units","semantic_units","semantic_kind","provider","crs"]
    rows=[{"attribute":k,"value":obj.attrs.get(k)} for k in keys if k in obj.attrs]
    if rows:
        display(pd.DataFrame(rows).style.hide(axis="index").set_caption("Selected source + semantic metadata"))

print("Python:",sys.version.split()[0])
print("CubeDynamics:",cd.__version__)
print("Imported from:",cd.__file__)
assert cd.__version__=="0.1.0rc3"


# Chapter 1 — What can CubeDynamics say?

Before telling a story, inspect the vocabulary. The point is not to memorize it. The point is that the software has a visible scientific language: nouns, source flavors, and verbs with semantic expectations.

In [ ]:
try:
    sources=data.list_sources()
    noun_table=pd.DataFrame(
        [{"noun":noun,"source":source} for noun,source_list in sources.items() for source in source_list]
    ).sort_values(["noun","source"]).reset_index(drop=True)
    display(noun_table.style.hide(axis="index").set_caption("Built-in environmental nouns and source flavors"))

    fig,ax=plt.subplots(figsize=(8,4))
    noun_table.groupby("noun").size().sort_values().plot(kind="barh",ax=ax)
    ax.set_title("Built-in environmental vocabulary")
    ax.set_xlabel("implemented source flavors")
    ax.set_ylabel("")
    plt.show()
except Exception as exc:
    panel("Vocabulary discovery",f"{type(exc).__name__}: {html.escape(str(exc))}","red")

print("Selected semantic verbs:")
print([n for n in dir(v) if n in [
    "anomaly","variance","zscore","threshold_state","quantile_state",
    "overlap","detect_events","occurrence_synchrony","severity_synchrony",
    "timing_synchrony","duration_synchrony","sync_with","plot"
]])


# Story 1 — Working Lands

<div class="cd-question"><b>Scientific question</b><br>
Where and when did unusually warm July days and days with little or no measured precipitation coincide in central South Dakota?
</div>

In [ ]:
BBOX_SD=[-101.2,43.7,-100.4,44.3]
START_SD,END_SD="2024-07-01","2024-07-31"

temperature=data.temperature(source="prism",statistic="maximum",bbox=BBOX_SD,start=START_SD,end=END_SD)
precipitation=data.precipitation(source="prism",bbox=BBOX_SD,start=START_SD,end=END_SD)

compact_attrs(temperature)
compact_attrs(precipitation)


## 1A · First look at the observations

In [ ]:
inds=[0,len(temperature.time)//2,-1]
labels=["July 1","Mid-July","July 31"]

fig,axes=plt.subplots(1,3,figsize=(15,4.5),constrained_layout=True)
for ax,idx,label in zip(axes,inds,labels):
    im=temperature.isel(time=idx).plot(ax=ax,add_colorbar=False)
    plt.colorbar(im,ax=ax,label=temperature.attrs.get("units",""))
    ax.set_title(label)
fig.suptitle("PRISM maximum temperature: the field changes through July",fontsize=17)
plt.show()

fig,axes=plt.subplots(1,2,figsize=(12,4.7),constrained_layout=True)
precipitation.isel(time=0).plot(ax=axes[0])
axes[0].set_title("One precipitation observation")
precipitation.sum("time").plot(ax=axes[1])
axes[1].set_title("Accumulated July precipitation")
plt.show()


## 1B · HTML cube: make the time dimension tangible

A static map hides the fact that the noun is a three-dimensional object. Let CubeDynamics render the real PRISM cube as an interactive object.


In [ ]:
prism_cube_html = safe(
    "CubeDynamics HTML cube · PRISM temperature",
    lambda: v.plot(
        temperature,
        title="PRISM maximum temperature · July 2024",
    )
)


## 1C · Ask a transformation question

<div class="cd-question"><b>Scientific question</b><br>
Where was maximum temperature most variable after removing each cell's July mean?
</div>

In [ ]:
temp_variability=(
    pipe(temperature)
    | v.anomaly(dim="time")
    | v.variance(dim="time",keep_dim=False)
)
semantic_report(temp_variability)

variance_raw=temp_variability.unwrap()
print("Raw dimensions:",variance_raw.dims)
print("Semantic dimensions:",getattr(temp_variability.semantic_state,"dimensions",None))

fig,ax=plt.subplots(figsize=(8.5,5.2))
variance_raw.plot(ax=ax)
ax.set_title("Variance of July temperature anomalies")
plt.show()


## 1D · Observations become explicit conditions

In [ ]:
warm_pipe=pipe(temperature)|v.quantile_state(quantile=.75,direction="above",name="warm_july_day")
dry_pipe=pipe(precipitation)|v.threshold_state(threshold=.1,direction="below",name="trace_or_no_rain")

warm=warm_pipe.unwrap()
dry=dry_pipe.unwrap()
warm_state=state_field(warm)
dry_state=state_field(dry)

day=min(15,warm_state.sizes["time"]-1)
fig,axes=plt.subplots(1,3,figsize=(15,4.4),constrained_layout=True)
warm_state.isel(time=day).astype(int).plot(ax=axes[0],vmin=0,vmax=1,add_colorbar=False)
axes[0].set_title("Warm condition")
dry_state.isel(time=day).astype(int).plot(ax=axes[1],vmin=0,vmax=1,add_colorbar=False)
axes[1].set_title("Dry condition")
warm_state.mean("time").plot(ax=axes[2],vmin=0,vmax=1)
axes[2].set_title("Warm frequency through July")
fig.suptitle("Observations become declared scientific conditions",fontsize=17)
plt.show()

text_panel("Warm condition",warm_pipe.explain())
text_panel("Dry condition",dry_pipe.explain())


## 1E · Compose the statement

<div class="cd-card"><div class="cd-title">Readable sentence</div>
<b>warm → overlap(dry) → mean(time)</b>
</div>


In [ ]:
try:
    _=pipe(warm)|v.overlap(dry,name="warm_and_dry")
except Exception as exc:
    panel(
        "rc3 temporal-support guardrail",
        f"{type(exc).__name__}: {html.escape(str(exc))}",
        "red",
    )

hot_dry=(
    pipe(warm)
    | v.overlap(
        dry,
        name="warm_and_dry",
        temporal_alignment="labels",
    )
    | v.mean(dim="time",keep_dim=False)
)
semantic_report(hot_dry)

freq=state_field(hot_dry.unwrap())
fig,ax=plt.subplots(figsize=(8.5,5.2))
(freq*100).plot(ax=ax,vmin=0,vmax=100)
ax.set_title("How often did warm + dry coincide?")
plt.show()


## 1F · HTML cube: view the joint condition through time

In [ ]:
joint_condition=(
    pipe(warm)
    | v.overlap(
        dry,
        name="warm_and_dry",
        temporal_alignment="labels",
    )
).unwrap()

joint_html=safe(
    "CubeDynamics HTML cube · warm ∩ dry",
    lambda: v.plot(
        state_field(joint_condition).astype(float),
        title="Warm and dry condition through July",
    )
)


# Story 2 — Boulder Cold Snap

<div class="cd-question"><b>Scientific question</b><br>
Can we move from an observed cold spell to a condition, events, spatial synchrony, and lagged relationships without losing the question?
</div>

In [ ]:
BBOX_CO=[-105.55,39.75,-104.85,40.35]
START_CO,END_CO="2024-01-01","2024-01-31"

cold_temp=data.temperature(source="prism",statistic="maximum",bbox=BBOX_CO,start=START_CO,end=END_CO)
cold_ppt=data.precipitation(source="prism",bbox=BBOX_CO,start=START_CO,end=END_CO)

regional_t=(pipe(cold_temp)|v.mean(dim=("y","x"),keep_dim=False)).unwrap().compute()
regional_p=(pipe(cold_ppt)|v.mean(dim=("y","x"),keep_dim=False)).unwrap().compute()

fig,ax=plt.subplots(figsize=(12,4.6))
regional_t.plot(ax=ax,marker="o",label="maximum temperature")
ax.axhline(0,linestyle="--",linewidth=1.2,label="0 °C")
ax2=ax.twinx()
regional_p.plot(ax=ax2,alpha=.45)
ax.set_title("January 2024: temperature and precipitation across Boulder region")
ax.set_ylabel("temperature")
ax2.set_ylabel("precipitation")
ax.legend(loc="upper left")
plt.show()


## 2A · HTML cube: the cold snap as a moving field

In [ ]:
cold_cube_html=safe(
    "CubeDynamics HTML cube · Boulder cold snap",
    lambda: v.plot(
        cold_temp,
        title="Boulder-region daily maximum temperature · January 2024",
    )
)


## 2B · Turn cold into a state

In [ ]:
cold=(
    pipe(cold_temp)
    | v.threshold_state(
        threshold=0,
        direction="below",
        name="freezing_day",
    )
)
semantic_report(cold)

cold_state=state_field(cold.unwrap())
fig,axes=plt.subplots(1,2,figsize=(12,4.5),constrained_layout=True)
cold_state.isel(time=13).astype(int).plot(ax=axes[0],vmin=0,vmax=1)
axes[0].set_title("Cold state on a January day")
cold_state.mean("time").plot(ax=axes[1],vmin=0,vmax=1)
axes[1].set_title("Fraction of January days below freezing")
plt.show()


## 2C · Ask whether cold happened together across space

In [ ]:
cold_occurrence=safe(
    "Occurrence synchrony",
    lambda: (
        pipe(cold.unwrap())
        | v.occurrence_synchrony(
            spatial_mode="reference",
            reference="center",
            method="jaccard",
        )
    )
)

if cold_occurrence is not None:
    semantic_report(cold_occurrence)
    occ=cold_occurrence.unwrap()
    if isinstance(occ,xr.Dataset) and "occurrence_synchrony" in occ:
        shown=occ["occurrence_synchrony"].squeeze()
        fig,ax=plt.subplots(figsize=(8.5,5.2))
        shown.plot(ax=ax,vmin=0,vmax=1)
        ax.set_title("How synchronous were freezing days with the center?")
        plt.show()


## 2D · HTML synchrony surface

In [ ]:
if cold_occurrence is not None:
    occ=cold_occurrence.unwrap()
    if isinstance(occ,xr.Dataset) and "occurrence_synchrony" in occ:
        synchrony_html=safe(
            "CubeDynamics HTML plot · occurrence synchrony",
            lambda: v.plot(
                occ["occurrence_synchrony"],
                title="Freezing-day occurrence synchrony",
            )
        )


## 2E · Conditions become events

In [ ]:
cold_events=safe(
    "Detect cold events",
    lambda: (
        pipe(cold.unwrap())
        | v.detect_events(
            min_duration=2,
            max_gap=0,
        )
    )
)

if cold_events is not None:
    semantic_report(cold_events)
    event_obj=cold_events.unwrap()
    display(event_obj.catalog.head(12) if hasattr(event_obj,"catalog") else event_obj)


## 2F · Event timing and duration synchrony

In [ ]:
timing_sync=None
duration_sync=None

if cold_events is not None:
    timing_sync=safe(
        "Timing synchrony",
        lambda: (
            pipe(cold_events.unwrap())
            | v.timing_synchrony(
                spatial_mode="neighbors",
                radius_km=75,
                match_tolerance="3D",
            )
        )
    )
    duration_sync=safe(
        "Duration synchrony",
        lambda: (
            pipe(cold_events.unwrap())
            | v.duration_synchrony(
                spatial_mode="neighbors",
                radius_km=75,
                match_tolerance="3D",
                min_matched_events=1,
            )
        )
    )

fig,axes=plt.subplots(1,2,figsize=(12,4.8),constrained_layout=True)
plotted=0
if timing_sync is not None:
    d=timing_sync.unwrap()
    if isinstance(d,xr.Dataset) and "timing_synchrony" in d:
        z=d["timing_synchrony"]
        if "time_window_end" in z.dims: z=z.isel(time_window_end=-1)
        z.squeeze().plot(ax=axes[0],vmin=0,vmax=1)
        axes[0].set_title("Cold-event timing synchrony")
        plotted+=1
if duration_sync is not None:
    d=duration_sync.unwrap()
    key="duration_similarity" if isinstance(d,xr.Dataset) and "duration_similarity" in d else None
    if key:
        z=d[key]
        if "time_window_end" in z.dims: z=z.isel(time_window_end=-1)
        z.squeeze().plot(ax=axes[1],vmin=0,vmax=1)
        axes[1].set_title("Cold-event duration similarity")
        plotted+=1
if plotted:
    plt.show()
else:
    plt.close(fig)


## 2G · Add a second noun and ask about lag

In [ ]:
wet=(
    pipe(cold_ppt)
    | v.threshold_state(
        threshold=1.0,
        direction="above",
        name="wet_day",
    )
)

fig,axes=plt.subplots(1,2,figsize=(11,4.4),constrained_layout=True)
state_field(cold.unwrap()).isel(time=13).astype(int).plot(ax=axes[0],vmin=0,vmax=1)
axes[0].set_title("Cold")
state_field(wet.unwrap()).isel(time=13).astype(int).plot(ax=axes[1],vmin=0,vmax=1)
axes[1].set_title("Wet")
plt.show()

cold_wet=safe(
    "Cold–wet lagged coupling",
    lambda: (
        pipe(cold.unwrap())
        | v.sync_with(
            wet.unwrap(),
            synchrony="occurrence",
            spatial_relation="same_pixel",
            lags=["0D","1D","2D","3D"],
        )
    )
)

if cold_wet is not None:
    semantic_report(cold_wet)
    coupling=cold_wet.unwrap()
    if isinstance(coupling,xr.Dataset) and "coupling_score" in coupling:
        score=coupling["coupling_score"]
        spatial_dims=[d for d in score.dims if d!="lag"]
        curve=score.median(dim=spatial_dims,skipna=True)
        fig,ax=plt.subplots(figsize=(8.5,4.5))
        ax.plot(coupling["lag"].values,curve.values,marker="o")
        ax.set_title("Cold–wet occurrence coupling by lag")
        ax.set_xlabel("wet-condition lag")
        ax.set_ylabel("median same-pixel coupling")
        ax.grid(alpha=.25)
        plt.show()


# Story 3 — Multivariate Weather

<div class="cd-question"><b>Scientific question</b><br>
Can the same grammar move across several weather nouns and still preserve what each quantity actually is?
</div>

In [ ]:
GRIDMET=[
    ("temperature",{"statistic":"maximum"}),
    ("precipitation",{}),
    ("vpd",{}),
    ("humidity",{"statistic":"maximum"}),
    ("wind",{}),
    ("radiation",{}),
]

gridmet={}
rows=[]
for noun,kwargs in GRIDMET:
    try:
        da=getattr(data,noun)(source="gridmet",bbox=BBOX_SD,start=START_SD,end=END_SD,**kwargs)
        gridmet[noun]=da
        rows.append({"noun":noun,"status":"loaded","shape":" × ".join(map(str,da.shape)),"units":da.attrs.get("units","")})
    except Exception as exc:
        rows.append({"noun":noun,"status":"failed","shape":"","units":str(exc)[:80]})

display(pd.DataFrame(rows).style.hide(axis="index").set_caption("Real gridMET noun retrieval"))


## 3A · One landscape, six environmental nouns

In [ ]:
preferred=[
    ("temperature","Temperature"),("precipitation","Precipitation"),("vpd","VPD"),
    ("humidity","Humidity"),("wind","Wind"),("radiation","Radiation")
]
available=[x for x in preferred if x[0] in gridmet]
nr=int(np.ceil(len(available)/3))
fig,axes=plt.subplots(nr,3,figsize=(15,4.2*nr),constrained_layout=True)
axes=np.atleast_1d(axes).ravel()
for ax,(noun,title) in zip(axes,available):
    da=gridmet[noun]
    idx=min(15,da.sizes["time"]-1)
    da.isel(time=idx).plot(ax=ax)
    ax.set_title(title)
for ax in axes[len(available):]:
    ax.axis("off")
fig.suptitle("One landscape, multiple built-in gridMET nouns",fontsize=18)
plt.show()


## 3B · Reuse one verb across unlike nouns

In [ ]:
zpipes={}
rows=[]
for noun,da in gridmet.items():
    try:
        zp=pipe(da)|v.zscore(dim="time")
        zpipes[noun]=zp
        rows.append({"noun":noun,"verb":"zscore(time)","status":"PASS"})
    except Exception as exc:
        rows.append({"noun":noun,"verb":"zscore(time)","status":f"FAIL: {type(exc).__name__}"})

display(pd.DataFrame(rows).style.hide(axis="index").set_caption("Same transformation, different nouns"))

items=list(zpipes.items())
if items:
    nr=int(np.ceil(len(items)/3))
    fig,axes=plt.subplots(nr,3,figsize=(15,4.2*nr),constrained_layout=True)
    axes=np.atleast_1d(axes).ravel()
    for ax,(noun,p) in zip(axes,items):
        np.abs(p.unwrap()).max("time").plot(ax=ax)
        ax.set_title(noun)
    for ax in axes[len(items):]:
        ax.axis("off")
    fig.suptitle("Strongest standardized July departure by environmental noun",fontsize=18)
    plt.show()


## 3C · Build a teaching multivariate index

In [ ]:
if all(k in zpipes for k in ("temperature","vpd","wind")):
    zt,zv,zw=xr.align(
        zpipes["temperature"].unwrap(),
        zpipes["vpd"].unwrap(),
        zpipes["wind"].unwrap(),
        join="inner",
    )
    weather_extremeness=((zt+zv+zw)/3).rename("weather_extremeness")
    weather_extremeness.attrs.update({
        "units":"1",
        "description":"Teaching index: mean z-score of temperature, VPD, and wind",
    })

    extreme_weather=(
        pipe(weather_extremeness)
        | v.quantile_state(
            quantile=.90,
            direction="above",
            name="upper_decile_weather_extremeness",
        )
    )
    semantic_report(extreme_weather)

    e=state_field(extreme_weather.unwrap())
    fig,axes=plt.subplots(1,2,figsize=(12,4.7),constrained_layout=True)
    e.mean("time").plot(ax=axes[0],vmin=0,vmax=1)
    axes[0].set_title("Where was unusual weather frequent?")
    e.mean(("y","x")).plot(ax=axes[1],marker="o")
    axes[1].set_title("When was unusual weather widespread?")
    plt.show()


## 3D · HTML cube: multivariate extremeness through time

In [ ]:
if "weather_extremeness" in globals():
    multi_html=safe(
        "CubeDynamics HTML cube · multivariate weather extremeness",
        lambda: v.plot(
            weather_extremeness,
            title="Temperature + VPD + wind · standardized extremeness",
        )
    )


## 3E · Same noun, different measurement

In [ ]:
if "temperature" in gridmet:
    p=(pipe(temperature)|v.mean(dim=("y","x"),keep_dim=False)).unwrap().compute()
    g=(pipe(gridmet["temperature"])|v.mean(dim=("y","x"),keep_dim=False)).unwrap().compute()

    # Convert gridMET K to °C for a fair visual comparison while keeping source identity explicit.
    g_c=g-273.15
    g_c.attrs["units"]="degC"

    common=np.intersect1d(p.time.values,g_c.time.values)
    p,g_c=p.sel(time=common),g_c.sel(time=common)

    fig,axes=plt.subplots(1,2,figsize=(12.5,4.7),constrained_layout=True)
    p.plot(ax=axes[0],marker="o",label="PRISM")
    g_c.plot(ax=axes[0],marker="s",label="gridMET converted to °C")
    axes[0].set_title("Regional daily trajectories")
    axes[0].legend()

    axes[1].scatter(p.values,g_c.values)
    lo=np.nanmin([p.values.min(),g_c.values.min()])
    hi=np.nanmax([p.values.max(),g_c.values.max()])
    axes[1].plot([lo,hi],[lo,hi],linestyle="--")
    axes[1].set_xlabel("PRISM °C")
    axes[1].set_ylabel("gridMET °C")
    axes[1].set_title("Same noun ≠ identical product")
    fig.suptitle("Compare source-qualified temperature without declaring equivalence",fontsize=17)
    plt.show()


# Story 4 — Remote Sensing

<div class="cd-question"><b>Scientific question</b><br>
Can the same scientific vocabulary move from meteorological cubes to repeated observations of vegetation?
</div>

In [ ]:
ndvi=safe(
    "Load Sentinel-2 vegetation index",
    lambda: data.vegetation_index(
        source="sentinel2",
        index="ndvi",
        lat=40.015,
        lon=-105.2705,
        start="2024-06-01",
        end="2024-06-30",
    )
)

if ndvi is not None:
    compact_attrs(ndvi)
    n=min(3,ndvi.sizes.get("time",1))
    if "time" in ndvi.dims and n>0:
        fig,axes=plt.subplots(1,n,figsize=(5*n,4.5),constrained_layout=True)
        axes=np.atleast_1d(axes)
        for i,ax in enumerate(axes):
            ndvi.isel(time=i).plot(ax=ax,vmin=-.2,vmax=1)
            ax.set_title(str(pd.Timestamp(ndvi.time.values[i]).date()))
        fig.suptitle("Sentinel-2 NDVI through a CubeDynamics noun",fontsize=17)
        plt.show()


## 4A · HTML cube: vegetation through repeated satellite observations

In [ ]:
if ndvi is not None:
    ndvi_html=safe(
        "CubeDynamics HTML cube · Sentinel-2 NDVI",
        lambda: v.plot(
            ndvi,
            title="Sentinel-2 NDVI through June 2024",
        )
    )


# Epilogue — What changed across the four stories?

The stories are deliberately different:

- **Working Lands**: observation → condition → overlap → summary
- **Cold Snap**: observation → condition → event → synchrony → lagged relationship
- **Multivariate Weather**: multiple nouns → standardization → composition → source comparison
- **Remote Sensing**: repeated satellite observations → vegetation cube

The point is not that these objects become interchangeable.

The point is that the **visible code keeps reading like the analysis while source, dimensions, semantics, and provenance remain inspectable**.


In [ ]:
rows=[]
for name in list(RESULTS):
    rows.append({"section":name,"status":"PASS","detail":""})
for name,detail in FAILURES.items():
    rows.append({"section":name,"status":"FAIL","detail":detail})

if rows:
    summary=pd.DataFrame(rows)
    def style_status(row):
        return ["background-color:#edf7ef"]*len(row) if row["status"]=="PASS" else ["background-color:#fff0f0"]*len(row)
    display(summary.style.apply(style_status,axis=1).hide(axis="index").set_caption("Supershowcase run summary"))

panel(
    "The story in one sentence",
    "<b>Four very different environmental questions were expressed with one readable analytical grammar, without pretending their data, states, events, relationships, or sources were the same thing.</b>",
    "green",
)


# Final takeaway

## Shorter code should not mean less inspectable science.

CubeDynamics' promise is not that PRISM, gridMET, Sentinel-2, thresholds, events, synchrony, and reductions become interchangeable.

It is that their scientific roles become easier to read while their distinctions remain available for inspection.

<div class="cd-grammar">noun → source → pipe → verb → state → trace → evidence</div>
